# 🐖 02 | Preparación de segmentación y particiones

**RECURSO REQUERIDO: CPU (sin acelerador)**. Este notebook realiza exclusivamente su propia etapa. **No cambies CPU/GPU dentro del notebook.**

Proyecto: Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖. Datos: FaroPigSeg, licencia reportada CC BY-NC 4.0; citar fuente https://github.com/yudong888/FaroPig-Datasets.

**Ruta general:** `MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖/`. Ejecutar en orden 00 → 01 → 02 → 03 → 04. Todos los resultados se guardan en Drive.

## Objetivo
Preparar las listas de imágenes `train`, `val`, `test` por **fecha completa inferida del nombre del archivo**, con semilla fija y sin copiar ni modificar los originales. También representa máscaras de referencia.

**Límite:** las fechas no garantizan independencia entre granjas/corrales/animales; es una mejora frente al split original por imagen.

In [ ]:
%pip -q install numpy pandas pillow matplotlib pyyaml openpyxl scikit-learn

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(
        'No se encontró la carpeta principal. Verifica la cuenta de Google y el nombre de la carpeta: '
        'Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
    )
DATA_ROOT = ROOT / '01_Datasets' / 'FaroPigSeg' / 'extraido' / 'FaroPigSeg'
DATASET_DIR = ROOT / '01_Datasets' / 'FaroPigSeg'
RES = ROOT / '03_Resultados'
EXPO = ROOT / '05_Exportaciones_ZIP'
for sub in ('01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento'):
    (RES / sub).mkdir(parents=True, exist_ok=True)
EXPO.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)

In [ ]:
import re,random,hashlib,yaml,numpy as np,pandas as pd
from PIL import Image
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon

base=DATA_ROOT
splits=('train','val','test')
items=[]; regex=re.compile(r'^(\d{4}-\d{2}-\d{2})')
for orig in splits:
    d=base/orig/'images'
    assert d.is_dir(),f'No existe {d}. Primero ejecuta 00.'
    for img in sorted(d.iterdir()):
        if img.suffix.lower() not in {'.jpeg','.jpg','.png','.webp','.bmp'}:continue
        match=regex.match(img.name)
        if not match:raise ValueError(f'Imagen sin fecha identificable: {img.name}')
        lbl=base/orig/'labels'/(img.stem+'.txt')
        if not lbl.is_file():raise FileNotFoundError(f'Falta la etiqueta {lbl}')
        items.append({'imagen':str(img),'origen':orig,'fecha':match.group(1),'etiqueta':str(lbl)})
assert items,'No hay imágenes.'
source_fingerprint=hashlib.sha256('\n'.join(sorted(z['imagen'] for z in items)).encode()).hexdigest()
DEST=DATASET_DIR/'split_fecha_v1'
DEST.mkdir(parents=True,exist_ok=True)
manifest=DEST/'particion_fecha_manifest.json'
if manifest.is_file():
    old=json.loads(manifest.read_text(encoding='utf8'))
    if old['fingerprint_lista_imagenes']!=source_fingerprint:
        raise RuntimeError('Cambió la lista de imágenes. Revisar manifiesto antes de regenerar particiones.')
    assignments=old['asignaciones_fecha']
    print('Reutilizando partición congelada previamente; no se sortean nuevas fechas.')
else:
    from sklearn.model_selection import GroupShuffleSplit
    fechas=sorted({x['fecha'] for x in items})
    if len(fechas)<10:raise ValueError('Muy pocas fechas para partición 3 vías.')
    gr=[x['fecha'] for x in items]
    idx=np.arange(len(items))
    g1=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=2026)
    remaining,test_indices=next(g1.split(idx,groups=gr))
    g2=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=2027)
    train_r,val_r=next(g2.split(remaining,groups=[gr[j] for j in remaining]))
    assignments={gr[j]:'train' for j in remaining[train_r]}
    assignments.update({gr[j]:'val' for j in remaining[val_r]})
    assignments.update({gr[j]:'test' for j in test_indices})
    manifest.write_text(json.dumps({'semilla':2026,'semilla_val':2027,'algoritmo':'GroupShuffleSplit test 20% fechas; validacion 20% restante', 'criterio':'fecha_iso_en_nombre','fingerprint_lista_imagenes':source_fingerprint,
                                    'asignaciones_fecha':assignments,'fecha_utc':datetime.now(timezone.utc).isoformat()},
                                   ensure_ascii=False,indent=2),encoding='utf8')
assert set(assignments)=={z['fecha'] for z in items}
by_split={s:sorted(x['imagen'] for x in items if assignments[x['fecha']]==s) for s in splits}
assert sum(map(len,by_split.values()))==len(items) and all(by_split[s] for s in splits)
for s in splits:(DEST/f'{s}.txt').write_text('\n'.join(by_split[s])+'\n',encoding='utf8')
conf={'path':str(base),'train':str(DEST/'train.txt'),'val':str(DEST/'val.txt'),
      'test':str(DEST/'test.txt'),'names':{0:'cerdo'}}
config=DEST/'data_fecha.yaml'
config.write_text(yaml.safe_dump(conf,sort_keys=False,allow_unicode=True),encoding='utf8')
registro=pd.DataFrame([{**x,'nuevo_split':assignments[x['fecha']]} for x in items])
resumen=registro.groupby('nuevo_split',as_index=False).agg(imagenes=('imagen','size'),fechas=('fecha','nunique'))
print('✅ Partición de fechas sin solapamientos:');print(resumen.to_string(index=False))
print('Config YOLO:',config)
print('Fechas por split:',{s:len(set(registro.loc[registro.nuevo_split.eq(s),'fecha'])) for s in splits})

In [ ]:
excel=RES/'01_Exploracion'/'CERDOS_02_Particion_Independiente_Fecha.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as xw:
    resumen.to_excel(xw,sheet_name='Resumen',index=False)
    registro.to_excel(xw,sheet_name='Asignaciones',index=False)
    pd.DataFrame([{'fecha':d,'particion':s} for d,s in sorted(assignments.items())]).to_excel(xw,sheet_name='Fechas',index=False)

fig,axs=plt.subplots(1,3,figsize=(15,5))
for ax,s in zip(axs,splits):
    img=Path(by_split[s][0]); lab=Path(str(img).replace('/images/','/labels/')).with_suffix('.txt')
    with Image.open(img) as im:
        rgb=np.array(im.convert('RGB'))
    ax.imshow(rgb);h,w=rgb.shape[:2];ax.set_title(f'{s}: anotaciones originales');ax.axis('off')
    for l in lab.read_text(encoding='utf-8-sig').splitlines():
        xy=np.asarray([float(v) for v in l.split()[1:]],dtype=float).reshape(-1,2)
        xy=xy*np.array([w,h]);ax.add_patch(Polygon(xy,closed=True,fill=False,linewidth=1.1))
fig.tight_layout();figpath=RES/'04_Figuras'/'CERDOS_02_Mascaras_Referencia.png';fig.savefig(figpath,dpi=160);plt.show()
print('✅ Excel:',excel,'\n✅ Figura:',figpath,'\n✅ YAML:',config)

**Siguiente:** abrir `03_CERDOS_Entrenamiento_YOLO11n_GPU.ipynb` y seleccionar **GPU T4** antes de ejecutarlo. Hasta ahora no se ha entrenado ningún modelo.